In [1]:
pip install deepseek ollama sentence-transformer gradio lanchain chromadb -q

ERROR: Could not find a version that satisfies the requirement sentence-transformer (from versions: none)
ERROR: No matching distribution found for sentence-transformer
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install ollama -q

Defaulting to user installation because normal site-packages is not writeable
  Using cached ollama-0.4.7-py3-none-any.whl (13 kB)
     |████████████████████████████████| 73 kB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 431 kB 10.2 MB/s eta 0:00:01
     |████████████████████████████████| 166 kB 6.1 MB/s eta 0:00:01
     |████████████████████████████████| 96 kB 10.8 MB/s eta 0:00:01
     |████████████████████████████████| 70 kB 8.5 MB/s  eta 0:00:01
     |████████████████████████████████| 78 kB 10.5 MB/s eta 0:00:01
     |████████████████████████████████| 58 kB 9.3 MB/s eta 0:00:011
     |████████████████████████████████| 1.8 MB 11.9 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install langchain langchain-community -q

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install chromadb -q

  distutils: /private/var/folders/sm/1984t7v920g25k3ccs_t5dhr0000gn/T/pip-build-env-ng_p6aew/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/sm/1984t7v920g25k3ccs_t5dhr0000gn/T/pip-build-env-ng_p6aew/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/sm/1984t7v920g25k3ccs_t5dhr0000gn/T/pip-build-env-ng_p6aew/normal'
  distutils: /private/var/folders/sm/1984t7v920g25k3ccs_t5dhr0000gn/T/pip-build-env-ng_p6aew/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/sm/1984t7v920g25k3ccs_t5dhr0000gn/T/pip-build-env-ng_p6aew/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/sm/1984t7v920g25k3ccs_t5dhr0000gn/T/pip-build-env-ng_p6aew/overlay'
You should 

In [6]:
pip install gradio -q

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [13]:
# pip install sentence_transformers -q

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install pymupdf -q

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 18.6 MB 4.1 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [50]:
# imports
import ollama
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import gradio as gr
import re
from concurrent.futures import ThreadPoolExecutor

## Preprocess and embed the text

In [51]:
# Step 1: Load the document using PyMuPDFLoader
loader = PyMuPDFLoader("/Users/aashidutt/Desktop/Foundations of llms.pdf")
documents = loader.load()

# Step 2: Split text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

# Step 3: Initialize Ollama embeddings
embedding_function = OllamaEmbeddings(model="deepseek-r1:7b")

# Step 4: Parallelize embedding generation
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)

with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

# Step 5: Recreate the collection 
client = Client(Settings())
client.delete_collection(name="foundations_of_llms")  # Delete any existing collection if needed
collection = client.create_collection(name="foundations_of_llms")

# Step 6: Add documents and embeddings to Chroma
for idx, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[{'id': idx}], 
        embeddings=[embeddings[idx]], 
        ids=[str(idx)]  # Ensure IDs are strings
    )

print("Embeddings stored successfully!")


Embeddings stored successfully!


## Set up the RAG pipeline

In [52]:
# initialize retriever using chroma collection

retriever = Chroma(collection_name = "foundations_of_llms", client = client, embedding_function=embedding_function).as_retriever()

def retrieve_context(question):
    results = retriever.get_relevant_documents(question)
    context = "\n\n".join([doc.page_content for doc in results])
    return context

## Query DeepSeek R1

In [53]:
def query_deepseek(question, context):

    # Format the input as a structured prompt
    formatted_promt = f"Question: {question}\n\nContext: {context}"

    # Send the prompt to DeepSeek-R1 using Ollama
    response = ollama.chat(
        model = "deepseek-r1:7b",
        messages = [{'role':'user', 'content': formatted_promt}]
    )
    
    # Extract and clean the response
    response_content = response['message']['content']
    final_answer = re.sub(r'<think>.*?</think>', '', response_content, flags = re.DOTALL).strip()
    return final_answer


## combine retrieval and generation steps


In [54]:
def rag_pipeline(question):

    # Retrieve context from the vector store
    context = retrieve_context(question)
    
    # Generate an answer using DeepSeek-R1
    answer = query_deepseek(question, context)
    return answer

In [46]:
def ask_question(question):
    # Run the RAG pipeline
    return rag_pipeline(question)

# Create a Gradio interface
interface = gr.Interface(
    fn=ask_question,
    inputs="text",
    outputs="text",
    title="RAG Chatbot: Foundations of LLMs",
    description="Ask any question about the Foundations of LLMs book. Powered by DeepSeek-R1."
)

# Launch the Gradio app
interface.launch(debug = True)


Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
